In [ ]:
# 匯入 json 模組用於解析 JSON 檔案
# 匯入 pprint 模組用於美化輸出資料結構
import json
from pprint import pprint

# 開啟並讀取空氣品質 JSON 檔案，使用 utf-8 編碼
with open("空氣品質aqi.json",encoding="utf-8") as file:
    data:dict = json.load(file)

# 從 JSON 資料中取出 'records' 陣列，包含所有測站的空氣品質資料
contents:list[dict]= data['records']

# 美化輸出查看資料結構
pprint(contents)

{'aqi': '78',
  'co': '0.76',
  'co_8hr': '0.4',
  'county': '新北市',
  'datacreationdate': '2024-03-15 09:00',
  'latitude': '24.94902778',
  'longitude': '121.38352778',
  'no': '25.4',
  'no2': '24.5',
  'nox': '50',
  'o3': '5.4',
  'o3_8hr': '9.8',
  'pm10': '44',
  'pm10_avg': '37',
  'pm2.5': '27',
  'pm2.5_avg': '26.6',
  'pollutant': '細懸浮微粒',
  'siteid': '311',
  'sitename': '新北(樹林)',
  'so2': '0.8',
  'so2_avg': '0',
  'status': '普通',
  'unit': '',
  'winddirec': '109',
  'windspeed': '0'}

In [11]:
# 從 pydantic 匯入 BaseModel、Field、field_validator
# 從 datetime 匯入 datetime 類別
from pydantic import BaseModel,Field,field_validator
from datetime import datetime

# 定義 AirSite 模型，對應單一測站的空氣品質資料
class AirSite(BaseModel):
    aqi:int                                    # 空氣品質指標
    county:str                                 # 縣市名稱
    date:datetime = Field(alias="datacreationdate")  # 資料建立時間，使用 alias 對應 JSON 欄位名
    lat:float = Field(alias = "latitude")      # 緯度
    lon:float = Field(alias="longitude")       # 經度
    pm25:float = Field(alias="pm2.5")          # PM2.5 濃度
    pollutant:str                              # 主要污染物名稱
    site_name:str = Field(alias="sitename")    # 測站名稱
    status:str                                 # 空品等級狀態

    #自定義驗證器：將空字串轉換為 0，避免型別轉換錯誤
    @field_validator('aqi','lat','lon','pm25', mode='before')
    @classmethod
    def empty_to_zero(cls, v):
        return 0 if v == '' else v


In [12]:
# 定義 Root 模型，作為整個 JSON 資料的根結構
class Root(BaseModel):
    status:bool = True          # 回應狀態，預設為 True
    sites:list[AirSite]         # 所有測站的列表

In [13]:
# 將每個 dict 解包並建立 AirSite 實例，組成列表後建立 Root 物件
root = Root(sites=[AirSite(**item) for item in contents])


In [ ]:
for site in root.sites:
    print(site)